
# Plane Crashes Data Profiling (1 - original dataset)

**Objective:** Profile and summarize the original dataset to assess data quality, completeness, and analytic readiness.


## LIBRARY


In [16]:
import os
import pandas as pd
from sqlalchemy import create_engine


## DIRECTORY PATH & DATABASE CONNECTION

In [ ]:
# Project directory
project_dir = os.path.join(os.path.expanduser("~"), "OneDrive", "Project_Code", "JobProject-MCCSS")


In [18]:
# Create SQLAlchemy engine
db_path = os.path.join(project_dir, "data", "raw", "plane_crashes_data.db")
engine = create_engine(f'sqlite:///{db_path}')

df = pd.read_sql_table("plane_crashes_data", engine)

# Close the connection
engine.dispose()

In [ ]:
# Dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5783 entries, 0 to 5782
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          5783 non-null   object
 1   time          5783 non-null   object
 2   location      5783 non-null   object
 3   operator      5783 non-null   object
 4   flight_no     5783 non-null   object
 5   route         5783 non-null   object
 6   ac_type       5783 non-null   object
 7   registration  5783 non-null   object
 8   cn_ln         5783 non-null   object
 9   aboard        5783 non-null   object
 10  fatalities    5783 non-null   object
 11  ground        5783 non-null   object
 12  summary       5783 non-null   object
dtypes: object(13)
memory usage: 587.5+ KB


<code style="color:orange">Notes</code>:

- Total rows: 5,783
- 13 columns:
    - No columns have missing values
        - Need to check if missing values are not assigned as something like `-`, ` `, `?`, `unknwon` or else
    - All columns are object or text (string), need to cheange the type of each field

Let's check a few rows.

In [30]:
df.sample(20)

,date,time,location,operator,flight_no,route,ac_type,registration,cn_ln,aboard,fatalities,ground,summary
3660,27-Nov-83,10:06,"Madrid-Barajas, Spain",AVIANCA,11,Paris - Madrid - Bogota,Boeing B-747-283B,HK-2910,21381,192 � (passengers:169� crew:23),181 � (passengers:158� crew:23),0,"While attempting to land at Madrid, the crew i..."
554,18-Dec-39,?,South of Gibraltar,Iberia Airlines,?,Alcante - Morocco,Junkers JU-52,M-CABA,5854,10 � (passengers:7� crew:3),10 � (passengers:7� crew:3),0,Shot down by British anti-aircraft fire. Crash...
427,6-Oct-36,?,Mexico,Mexicana,?,?,Lockheed Orion,XA-BDH,?,1 � (passengers:0� crew:1),1 � (passengers:0� crew:1),0,?
3097,9-May-76,16:30,"Near Cuneca, Spain",Military - Imperial Iranian Air Force,228,"Terhan, Iran - Madrid, Spain - McGuire AFB, Ne...",Boeing B-747-131F,5-283,19677/73,17 � (passengers:7� crew:10),17 � (passengers:7� crew:10),0,The aircraft was struck by lightning while des...
1995,4-Apr-63,4:30,"Tatarstan, Russia",Operator,25,Moscow - :\tKrasnoyarsk,Ilyushin IL-18,CCCP-75866,183005901,67 � (passengers:59� crew:8),67 � (passengers:59� crew:8),0,Crashed in a snow covered field whille en rout...
4044,15-Mar-89,7:25,"West Lafayette, Indiana",Mid Pacific Air,?,Terre Haute - Lafayette,NAMC YS-11A-300F,N128MP,2139,2 � (passengers:0� crew:2),2 � (passengers:0� crew:2),0,While landing the cargo plane pitched down and...
5448,1-Sep-08,12:10,"Columbus, Ohio",Air Tahoma,?,Colombus - Mansfield,Convair CV-580,N587X,361,3 � (passengers:0� crew:3),3 � (passengers:0� crew:3),0,Soon after taking off the pilot radioed he was...
4188,21-Nov-90,11:15,"Koh Samui, Thailand",Bangkok Airways,125,Bangkok - Koh Samui,de Havilland Canada DHC-8-103,OB-1358,T210-63675,38 � (passengers:33� crew:5),38 � (passengers:33� crew:5),0,"After receiving a runway change, the crew exec..."
2044,15-Feb-64,?,"Detroit, Michigan",Commercial Air Taxi,?,Detroit - Akron,Piper Aero Commander 560E,N3823C,?,4 � (passengers:3� crew:1),4 � (passengers:3� crew:1),0,Disapperared en route. Missing aircraft not re...
1104,1-Nov-49,c 11:45,"Arlington, Virginia",Eastern Air Lines / Military - Bolivian Air Force,537,Boston - Washington D.C. - New Orleans,Douglas C-54B / P-38,N88727/NX26927,18365 /,55 � (passengers:51� crew:4),55 � (passengers:51� crew:4),0,Midair collision. The P-38 hit the airliner fr...


<code style="color:orange">Notes</code>:

- Issues start popping up:
    - **Date** column: 
        - not formated as a date
        - year with 2 digits can cause problems for trend analysis

    - **Time** column: 
        - not formated as time; 
        - `?` suggesting this might be the `NULL` of the dataset; 
        - `c`in some cells, indicating we might get letters and symbols in this column, not only digits and a `:`.
    
    - **Location** column: 
        - no clear structure: `City, Country`, `State, Country`, `City, State, Country`
        - verify if `,` separator is consistent accross the data 
    
    - **Operator** column:
        - Ffree-text, might have spelling mistakes

    - **Flight No** colum:
        - seems to have a lot of missings (`?`)
        - verify if it is a number for the populated cells, might need change format
    
    - **Route** column:
        - multiple places, indicating trajectory with layovers (?)
        - verify if `-` separator is consistent accross the data 
        - inconsistent naming `City` or `City, ST`

    - **Aircraft Type** column:
        - can it be separated into make and model? (e.g. "Boeing B-747-283B" ==> Make: Boeing , Model: 747)
        - verify if `/` is a separator

    - **Registration** column:
        - free-text
        - check if cab be make consistent by removing `-`

    - **Construction Number** column:
        - has letters, `-`, `/`
        - verify if `/` is a separator

    - **Aboard** column:
        - aggregated text, has total number of people aboard, then how many were crew and how many were passengers
        - verify if total = passengers + crew numbers
        - create 3 columns for each information and format as integer

    - **Fatalities** column:
        - aggregated text, has total number of fatalities, then how many were crew and how many were passengers
        - verify if total = passengers + crew numbers
        - create 3 columns for each information and format as integer
    
    - **Ground** column:
        - count of people that died on the ground beyond who was on the aircraft
        - should be a integer
        - if missing, can it be 0?

    - **Summary** column:
        - free-text
        - look for spelling mistakes
        - can be use to get the cause of the crash?

Using this summary we can start cleaning the data so we can than working on a full data profiling.